In [ ]:
import pandas as pd
import re

In [ ]:
# 读取文件
excel_file1 = pd.ExcelFile('my_h_dish_category.xlsx')
excel_file2 = pd.ExcelFile('my_h_dish_info_all.xlsx')

# 获取指定工作表中的数据
df1 = excel_file1.parse('my_h_dish_category')
df2 = excel_file2.parse('my_h_dish_info_all')

In [ ]:
# 删除dish_name为空的数据行
df2 = df2.dropna(subset=['dish_name'])
# 查看数据基本信息
print("df1形状:", df1.shape)
print("df2形状:", df2.shape)

In [ ]:
# 从macronutrients列中提取各项信息
df2['energy'] = df2['macronutrients'].str.extract(r'能量(\d+% NRV\d+\.?\d* kcal)')
df2['protein'] = df2['macronutrients'].str.extract(r'蛋白质(\d+% NRV\d+\.?\d* g)')
df2['fat'] = df2['macronutrients'].str.extract(r'脂肪(\d+% NRV\d+\.?\d* g)')
df2['carbohydrates'] = df2['macronutrients'].str.extract(r'碳水化合物(\d+% NRV\d+\.?\d* g)')

# 查看提取结果
df2[['energy', 'protein', 'fat', 'carbohydrates']].head()

In [ ]:
# 处理energy列格式
df2['energy'] = (df2['energy']
                .str.replace(r'(%)(\s+)', r'\1', regex=True)
                .str.replace(r'(NRV)(\d)', r'\1 \2', regex=True)
                .str.replace(r'(\d)(\s+)(kcal)', r'\1\3', regex=True))

# 处理其他宏量营养素列格式
for col in ['protein', 'fat', 'carbohydrates']:
    df2[col] = (df2[col]
                .str.replace(r'(%)(\s+)', r'\1', regex=True)
                .str.replace(r'(NRV)(\d)', r'\1 \2', regex=True)
                .str.replace(r'(\d)(\s+)(g)', r'\1\3', regex=True))

# 查看格式化结果
df2[['energy', 'protein', 'fat', 'carbohydrates']].head()

In [ ]:
df2.head()

In [ ]:
# 提取能量列纯卡路里部分
df2['pure_calorie'] = df2['energy'].str.extract(r'(\d+\.?\d*kcal)')
df2['pure_calorie'].head()

In [ ]:
# 执行左连接操作，匹配分类信息
for index, row in df2.iterrows():
    # 匹配条件：dish_name相同且纯热量值一致
    match = df1[(df1['dish_name'] == row['dish_name']) & 
                (df1['calorie'] == row['pure_calorie'])]
    if not match.empty:
        df2.at[index, 'category'] = match.iloc[0]['category']
    else:
        df2.at[index, 'category'] = '/'

df2 = df2.drop(columns=['macronutrients', 'pure_calorie'])

In [ ]:
df2[['dish_name', 'energy', 'category']].head()

In [ ]:
# 合并df2的vitamin和minerals列，以\n的形式拼接
df2['vitamin_minerals_combined'] = df2['vitamin'].fillna('') + '\n' + df2['minerals'].fillna('')

In [ ]:
# 去除NRV的百分比和后边的克数等内容
def process_text(text):
    text = text.replace('维生素B6', '维生素B₆')
    text = text.replace('维生素B12', '维生素B₁₂')
    return text
# 应用函数到合并列
df2['vitamin_minerals_combined'] = df2['vitamin_minerals_combined'].apply(process_text)

In [ ]:
df2['vitamin_minerals_combined'].head().tolist()

In [ ]:
df2[['vitamin', 'minerals']].head()

In [ ]:
# 专门用于提取分类时去除NRV和单位的函数
all_categories = []
def clean_for_category(text):
    # 去除NRV的百分比和克数等内容
    cleaned = re.sub(r'\d+% NRV|\d+\.?\d* (μg RAE|mg α-TE|mg|μg|kcal|g)', '', text)
    return cleaned.strip()
# 提取所有分类
for item in df2['vitamin_minerals_combined']:
    categories = item.split('\n')
    for category in categories:
        # 对每个分类单独清理，不影响原始列
        cleaned_category = clean_for_category(category)
        if cleaned_category and cleaned_category not in all_categories:
            all_categories.append(cleaned_category)

print('所有分类：', all_categories)

In [ ]:
df2['vitamin_minerals_combined'].head().tolist()

In [ ]:
# 提取出所有分类如下
cn_to_en = {
    "维生素A": "vitamin_A",
    "维生素E": "vitamin_E",
    "硫胺素": "thiamine",
    "核黄素": "riboflavin",
    "维生素B₆": "vitamin_B₆",
    "维生素B₁₂": "vitamin_B₁₂",
    "烟酸": "niacin",
    "叶酸": "folic_acid",
    "维生素C": "vitamin_C",
    "生物素": "biotin",
    "总胆碱": "total_choline",
    "维生素D": "vitamin_D",
    "维生素K": "vitamin_K",
    "泛酸": "pantothenic_acid",
    "钠": "sodium",
    "钾": "potassium",
    "镁": "magnesium",
    "铁": "iron",
    "锌": "zinc",
    "钙": "calcium",
    "磷": "phosphorus",
    "硒": "selenium",
    "碘": "iodine",
    "铜": "copper",
    "锰": "manganese"
}

In [ ]:
# 创建英文列名的空列
for en_col in cn_to_en.values():
    df2[en_col] = ""
# 正则表达式用于提取数值部分（包含数字、单位和NRV信息）
pattern = re.compile(r'([^\d]+)(\d.+)')

# 处理每一行数据
for idx, row in df2.iterrows():
    combined_text = row['vitamin_minerals_combined']
    if not combined_text:
        continue
        
    # 按换行符拆分每个元素
    elements = combined_text.split('\n')
    for elem in elements:
        elem = elem.strip()
        if not elem:
            continue
            
        # 尝试匹配分类名称和数值部分
        match = pattern.match(elem)
        if match:
            category_cn = match.group(1).strip()  # 中文分类名称
            value = match.group(2).strip()        # 数值部分（包含单位和NRV）
            
            # 查找对应的英文列名
            if category_cn in cn_to_en:
                en_col = cn_to_en[category_cn]
                df2.at[idx, en_col] = value

In [ ]:
# 显示处理结果示例（前5行的部分列）
df2[['vitamin_minerals_combined', 'vitamin_A', 'vitamin_C', 'sodium', 'calcium']].head()

In [ ]:
# 对所有营养元素列应用格式处理
for en_col in cn_to_en.values():
    # 执行三项替换操作，处理不同单位格式
    df2[en_col] = df2[en_col].str.replace(r'(%)(\s+)', r'\1', regex=True)          # 去除%后的空格
    df2[en_col] = df2[en_col].str.replace(r'(NRV)(\d)', r'\1 \2', regex=True)      # 在NRV和数字间加空格
    df2[en_col] = df2[en_col].str.replace(r'(\d)(\s+)(μg RAE|mg α-TE|mg|μg|kcal|g)', r'\1\3', regex=True) # 去除kcal前的空格

In [ ]:
# 显示处理结果示例（前5行的部分列）
df2[['vitamin_minerals_combined', 'vitamin_A', 'vitamin_C', 'sodium', 'calcium']].head()

In [ ]:
df2.head()

In [ ]:
def extract_num_unit(text):
    if pd.isna(text):  # 先处理NaN值，避免后续报错
        return ""
    # 正则表达式：匹配“数字（含小数）+ 单位”，支持中文单位（克/毫升/升）和英文单位（mg/μg/ml/L）
    # [克毫升升mgμgmlL] → 匹配常见单位（不区分大小写，通过flags=re.IGNORECASE实现）
    match = re.search(r'(\d+\.?\d*\s*[克毫升升mgμgmlL])', str(text), flags=re.IGNORECASE)
    return match.group(1).strip() if match else ""  # 有匹配则返回结果，无匹配返回空
# 应用函数处理列
df2['measurement_unit'] = df2['measurement_unit'].apply(extract_num_unit)
# 先将NaN填充为空字符串，再替换指定无效值
df2['quantity'] = df2['quantity'].fillna('').str.replace(r'.*未获取到单位量.*','',regex=True)
df2['cooking_method'] = df2['cooking_method'].fillna('').str.replace(r'.*未获取到数据.*','',regex=True)

In [ ]:
df2[['measurement_unit', 'quantity', 'cooking_method']].iloc[::-1]

In [ ]:
df2[['vitamin_minerals_combined', 'vitamin', 'minerals', 'img_url', 'img_path']]

In [ ]:
# 删除无用列
df2 = df2.drop(columns=['vitamin_minerals_combined', 'vitamin', 'minerals', 'img_url', 'img_path'])

In [ ]:
print(df2.columns.tolist()) 

In [ ]:
# 将结果保存为 Excel 文件
output_path = 'my_h_dish_info_alldata.xlsx'
df2.to_excel(output_path, index=False)